# E-Commerce Lakehouse: Bronze Orders


## Load the source orders

Read the original Olist orders CSV from our Unity Catalog Volume.

The first row contains column names, so we enable the header option.
We initially keep columns as strings to preserve the source values;
data types will be validated and converted in the Silver layer.

In [ ]:
raw_orders_df = (
spark.read
.option("header", True)
.csv("/Volumes/workspace/ecommerce/raw_files/olist_orders_dataset.csv")
)

## Preview the orders data

Display five rows to inspect the column names and sample values before further processing.

In [ ]:
display(raw_orders_df.limit(5))

## Inspect the source structure and row count

Count the source records and inspect the column data types.
These results establish a baseline for validating the Bronze table.

In [ ]:
print("Source order count:", raw_orders_df.count())
raw_orders_df.printSchema()

## Add ingestion metadata

Preserve the original orders data and add an ingestion_timestamp
column to record when the data is processed by our pipeline.

In [ ]:
from pyspark.sql.functions import current_timestamp

### Preview the Bronze orders

Inspect five rows to confirm that the original order fields remain
and the ingestion_timestamp column has been added.

In [ ]:
bronze_orders_df = raw_orders_df.withColumn("ingestion_timestamp", current_timestamp())

### Preview the Bronze orders

Display five rows to inspect the original order data and the added
ingestion timestamp.

In [ ]:
display(bronze_orders_df.limit(5))

## Refresh the Bronze orders table

Rebuild workspace.ecommerce.bronze_orders from the complete source CSV.
Overwrite replaces existing contents instead of appending duplicate orders.
Verify that the saved row count matches the source.

In [ ]:
table_name = "workspace.ecommerce.bronze_orders"

source_count = raw_orders_df.count()

if source_count != 99441:
    raise ValueError(
        f"Expected 99,441 source orders; found {source_count}. "
        "Stopped before replacing the table."
    )

(
    bronze_orders_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_count = spark.table(table_name).count()

if saved_count != source_count:
    raise ValueError("Saved Bronze count does not match the source.")

print("Bronze orders refreshed:", saved_count)

## Validate the saved Bronze table

Read the saved Delta table and compare its row count with the
99,441 records in the source CSV.

In [ ]:
saved_bronze_orders_df = spark.table("workspace.ecommerce.bronze_orders")

In [ ]:
print(saved_bronze_orders_df.count())